# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset—Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya—using the `mlcroissant` library.

### Dataset Source
The dataset metadata is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) at the provided URL. All data and schema references in this notebook use Croissant entity `@id`s to ensure unambiguous identification.


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. The Croissant schema provides the machine-readable definition of data record sets, fields, and the dataset's context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL (FAIR^2 dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset via Croissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # .to_json() not needed here; access as object

print(f"Dataset Name: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}")


## 2. Data Overview
List all available record sets (tables), their `@id`s, and enumerate fields and columns by their `@id`. This reveals the data's logical and physical organization for structured extraction and manipulation.

In [ ]:
# Examine available record sets by @id
record_sets_info = dataset.record_sets
if not record_sets_info:
    print("No record sets were found in the Croissant schema.")
else:
    for rs in record_sets_info:
        print(f"RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field']
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                print(f"  Field @id: {field.get('@id','<missing>')}")
                if 'column' in field:
                    columns = field['column']
                    if isinstance(columns, dict):
                        columns = [columns]
                    for col in columns:
                        print(f"    Column @id: {col.get('@id', '<missing>')}")
        print()


## 3. Data Extraction
Load records from selected record sets into Pandas DataFrames using their `@id`. If more than one record set is present, all are loaded into a dictionary by `@id`. All DataFrame columns are referenced by their field or column `@id`.

In [ ]:
# List all record set @ids found
record_sets = [rs['@id'] for rs in dataset.record_sets] if dataset.record_sets else []

dataframes = {}
for record_set_id in record_sets:
    # Records generator yields dictionaries with keys = Croissant field or column @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

if record_sets:
    # Show available columns by @id for the first record set
    first_set_id = record_sets[0]
    print(f"Columns in record set '@id' {first_set_id}:")
    print(dataframes[first_set_id].columns.tolist())
    display(dataframes[first_set_id].head())
else:
    print("No data tables could be extracted; please check the Croissant schema for available record sets.")


## 4. Exploratory Data Analysis (EDA)
For demonstration, select a numeric field from the first available record set (by `@id`).

We will:
- Filter records based on a numeric threshold
- Normalize the selected numeric field
- (If possible) Group data by a suitable categorical field


In [ ]:
# This cell will automatically pick a numeric field for demo purposes
import numpy as np

if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is not None:
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as demo threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Try grouping by a categorical/binary field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == object or df[col].dtype.name.startswith('category')):
                unique_vals = df[col].nunique(dropna=True)
                if unique_vals > 1 and unique_vals < 20:
                    group_field = col
                    break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields detected in the record set.")
else:
    print("No data for EDA.")


## 5. Visualization
Visualize the distribution of the selected numeric field and (if applicable) show group means for a chosen categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(
            data=grouped_df,
            x=group_field,
            y=numeric_field_id,
        )
        plt.title(f"Mean of {numeric_field_id} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
This notebook demonstrated:
- How to discover and access dataset structure using the Croissant schema and entity `@id`s
- Programmatic loading and display of record sets with `mlcroissant`
- Data transformation, normalization, and basic groupwise analysis referencing only Croissant `@id`s
- Visualization of selected field distributions for further insight

Continue to analyze and interpret the FAIR^2 data by leveraging its rich metadata and record set definitions, always using `@id` references for clarity and reproducibility.